# PhysIQ — Physics Reasoning Benchmark

**250 scenarios × 3 representation formats = 750 evaluation instances**

Five tasks of increasing cognitive complexity. Ground truth is computed by pymunk (deterministic, 60 Hz simulation), making scoring fully objective. All tasks scored 0–1 with partial credit.

| # | Task | Scenarios | Type |
|---|------|-----------|------|
| 1 | Trajectory Prediction | 60 | Single-turn |
| 2 | Stability Judgment | 60 | Single-turn |
| 3 | Causal Chain Reasoning | 60 | Single-turn |
| 4 | Tool Use Planning | 40 | Multi-turn (≤10 turns) |
| 5 | Adaptive Replanning | 30 | Multi-turn + forced perturbation |

**PhysIQ Score** = equal-weighted mean across all 5 tasks (0.20 each)  
**Format Robustness Score (FRS)** = `1 - (max - min) / max` across JSON / ASCII / NL (1.0 = ideal)

In [ ]:
import sys, os, json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'Kaggle_agi_bench'))

import kaggle_benchmarks as kbench
print('kaggle-benchmarks', kbench.__version__)

MASTER_SEED = 42
MAX_TURNS   = 10
OUTPUTS_DIR = 'outputs'
os.makedirs(OUTPUTS_DIR, exist_ok=True)

try:
    llm = kbench.llm
    print('LLM:', type(llm).__name__)
except AttributeError:
    llm = None
    print('No Kaggle LLM — eval cells will be skipped.')

In [ ]:
from physiq.engine     import PhysIQWorld
from physiq.formats    import build_prompt, format_as_json, format_as_ascii, format_as_nl
from physiq.generation import compute_ground_truth, validate_scenario, _task_seed
from physiq.scoring    import (
    score_trajectory, score_stability, score_causal_chain,
    score_tool_use, score_replan, TASK_WEIGHTS,
)
from physiq.templates               import SCENARIO_COUNTS
from physiq.templates.trajectory    import TRAJECTORY_TEMPLATES
from physiq.templates.stability     import STABILITY_TEMPLATES
from physiq.templates.causal_chain  import CAUSAL_CHAIN_TEMPLATES
from physiq.templates.tool_use      import TOOL_USE_TEMPLATES
from physiq.templates.replan        import REPLAN_TEMPLATES

print('physiq OK | scenario counts:', SCENARIO_COUNTS)

---
## Task 1: Trajectory Prediction

Given a 2D physics scenario, predict the final (x, y) position of a target object after the simulation runs.

**Answer format:** `ANSWER: [x, y]`  
**Scoring:** Continuous 0–1 based on normalised Euclidean distance to ground truth.

In [ ]:
task_type = 'trajectory'
counts    = SCENARIO_COUNTS[task_type]
task_seed = _task_seed(task_type, MASTER_SEED)
rng       = np.random.RandomState(task_seed)

scenarios_t1 = []
for difficulty in ['easy', 'medium', 'hard']:
    target = counts[difficulty]
    collected, attempt = [], 0
    while len(collected) < target and attempt < target * 3:
        seed     = int(rng.randint(0, 2**31))
        template = TRAJECTORY_TEMPLATES[attempt % len(TRAJECTORY_TEMPLATES)]
        attempt += 1
        try:
            s = template.generate(difficulty, seed)
        except Exception:
            continue
        if validate_scenario(s, task_type):
            s['_ground_truth'] = compute_ground_truth(s, task_type)
            collected.append(s)
    scenarios_t1.extend(collected)
    print(f'  T1 {difficulty}: {len(collected)}/{target}')

rows = []
for s in scenarios_t1:
    gt = s['_ground_truth']
    for fmt in ['json', 'ascii', 'nl']:
        rows.append({
            'scenario_id':    s['id'],
            'difficulty':     s['difficulty'],
            'representation': fmt,
            'prompt':         build_prompt(s, fmt, task_type),
            'ground_truth':   json.dumps(gt),
        })
df_trajectory = pd.DataFrame(rows)
print(f'df_trajectory: {df_trajectory.shape}')

In [ ]:
def _parse_trajectory_answer(response: str):
    m = re.search(r'ANSWER\s*:\s*\[([\d.\-+eE\s,]+)\]', response, re.IGNORECASE)
    if m:
        try:
            nums = [float(x) for x in m.group(1).split(',')]
            if len(nums) == 2:
                return nums
        except ValueError:
            pass
    for p in reversed(re.findall(r'\[([\d.\-+eE\s,]+)\]', response)):
        try:
            nums = [float(x) for x in p.split(',')]
            if len(nums) == 2:
                return nums
        except ValueError:
            pass
    return None


def _score_trajectory(response: str, ground_truth_json: str) -> float:
    gt        = json.loads(ground_truth_json)
    predicted = _parse_trajectory_answer(response)
    if predicted is None:
        return 0.0
    return score_trajectory(predicted, gt['final_position'], gt['world_diagonal'])


@kbench.task(
    name='PhysIQ Trajectory Prediction',
    description=(
        'Given a 2D physics scenario (JSON, ASCII art, or natural language), '
        'predict the final (x, y) position of the target object. '
        'Answer format: ANSWER: [x, y]. Tests genuine mental simulation.'
    ),
)
def physiq_trajectory(llm: kbench.LLMChat, prompt: str, ground_truth: str) -> float:
    """Predict final object position in 2D physics simulation."""
    return _score_trajectory(llm.prompt(prompt), ground_truth)


# Sanity check
gt0  = json.loads(df_trajectory.iloc[0]['ground_truth'])
fake = f"ANSWER: {gt0['final_position']}"
print('T1 perfect score:', _score_trajectory(fake, df_trajectory.iloc[0]['ground_truth']))

In [ ]:
if llm is not None:
    rows = []
    for _, row in df_trajectory.iterrows():
        run = physiq_trajectory.run(llm=llm, prompt=row['prompt'], ground_truth=row['ground_truth'])
        rows.append({'scenario_id': row['scenario_id'], 'difficulty': row['difficulty'],
                     'representation': row['representation'], 'score': run.result})
    df_t1 = pd.DataFrame(rows)
    df_t1.to_csv(f'{OUTPUTS_DIR}/task1_trajectory_results.csv', index=False)
    print(f'T1 mean: {df_t1["score"].mean():.3f}')
    print(df_t1.groupby(['difficulty', 'representation'])['score'].mean().unstack())
else:
    df_t1 = None
    print('Skipped.')

---
## Task 2: Stability Judgment

Given a 2D arrangement of objects, determine whether it will remain stable or collapse.

**Answer format:**
```
STABLE: yes/no
FIRST_FAILURE: <object_id>
FAILURE_DIRECTION: left/right/topples
```
**Scoring:** Judgment (0.5) + failure ID (0.3) + direction (0.2)

In [ ]:
task_type = 'stability'
counts    = SCENARIO_COUNTS[task_type]
task_seed = _task_seed(task_type, MASTER_SEED)
rng       = np.random.RandomState(task_seed)

scenarios_t2 = []
for difficulty in ['easy', 'medium', 'hard']:
    target = counts[difficulty]
    collected, attempt = [], 0
    while len(collected) < target and attempt < target * 3:
        seed     = int(rng.randint(0, 2**31))
        template = STABILITY_TEMPLATES[attempt % len(STABILITY_TEMPLATES)]
        attempt += 1
        try:
            s = template.generate(difficulty, seed)
        except Exception:
            continue
        if validate_scenario(s, task_type):
            s['_ground_truth'] = compute_ground_truth(s, task_type)
            collected.append(s)
    scenarios_t2.extend(collected)
    print(f'  T2 {difficulty}: {len(collected)}/{target}')

rows = []
for s in scenarios_t2:
    gt = s['_ground_truth']
    for fmt in ['json', 'ascii', 'nl']:
        rows.append({
            'scenario_id':    s['id'],
            'difficulty':     s['difficulty'],
            'representation': fmt,
            'prompt':         build_prompt(s, fmt, task_type),
            'ground_truth':   json.dumps(gt),
        })
df_stability = pd.DataFrame(rows)
print(f'df_stability: {df_stability.shape}')

In [ ]:
_FAILURE_KWS = {
    'left', 'right', 'topple', 'topples', 'fall', 'falls',
    'slide', 'slides', 'collapse', 'collapses', 'tip', 'tips',
    'rotate', 'rotates', 'overturn', 'lean', 'leans',
}


def _parse_stability(response: str) -> dict:
    resp = response.lower()
    sm   = re.search(r'stable\s*:\s*(yes|no|true|false|stable|unstable)', resp)
    if sm:
        is_stable = sm.group(1) in ('yes', 'true', 'stable')
    else:
        is_stable = 'unstable' not in resp and 'collapse' not in resp and 'fall' not in resp
    ff  = re.search(r'first_failure\s*:\s*([\w_]+)', resp)
    fd  = re.search(r'failure_direction\s*:\s*(\w+)', resp)
    direction = fd.group(1) if fd else next(
        (kw for kw in ['left', 'right', 'topples', 'topple'] if kw in resp), None
    )
    return {
        'stable':            is_stable,
        'first_failure':     ff.group(1) if ff else None,
        'failure_direction': direction,
    }


def _score_stability(response: str, ground_truth_json: str) -> float:
    gt    = json.loads(ground_truth_json)
    p     = _parse_stability(response)
    score = 0.5 if p['stable'] == gt.get('stable', True) else 0.0
    if gt.get('stable', True):
        return score
    evts = gt.get('failure_events', [])
    if evts and p['first_failure']:
        gt_obj = evts[0].get('object_id', '')
        if p['first_failure'].lower() in gt_obj.lower() or gt_obj.lower() in p['first_failure'].lower():
            score += 0.15
        gt_dir = evts[0].get('direction', '')
        if p['failure_direction'] and gt_dir and p['failure_direction'] in gt_dir:
            score += 0.15
    score += min(0.2, sum(1 for kw in _FAILURE_KWS if kw in response.lower()) * 0.05)
    return min(score, 1.0)


@kbench.task(
    name='PhysIQ Stability Judgment',
    description=(
        'Given a 2D arrangement of physics objects, determine whether it will '
        'remain stable or collapse. Tests spatial reasoning about balance and support.'
    ),
)
def physiq_stability(llm: kbench.LLMChat, prompt: str, ground_truth: str) -> float:
    """Predict stability of a 2D physics arrangement."""
    return _score_stability(llm.prompt(prompt), ground_truth)


gt0  = json.loads(df_stability.iloc[0]['ground_truth'])
fake = 'STABLE: yes' if gt0['stable'] else 'STABLE: no\nFIRST_FAILURE: obj\nFAILURE_DIRECTION: left\nblock topples left'
print('T2 sanity:', _score_stability(fake, df_stability.iloc[0]['ground_truth']))

In [ ]:
if llm is not None:
    rows = []
    for _, row in df_stability.iterrows():
        run = physiq_stability.run(llm=llm, prompt=row['prompt'], ground_truth=row['ground_truth'])
        rows.append({'scenario_id': row['scenario_id'], 'difficulty': row['difficulty'],
                     'representation': row['representation'], 'score': run.result})
    df_t2 = pd.DataFrame(rows)
    df_t2.to_csv(f'{OUTPUTS_DIR}/task2_stability_results.csv', index=False)
    print(f'T2 mean: {df_t2["score"].mean():.3f}')
else:
    df_t2 = None
    print('Skipped.')

---
## Task 3: Causal Chain Reasoning

Given a multi-body 2D physics scenario with a trigger event, describe the chain of interactions and the final outcome.

**Answer format:**
```
STEPS:
1. <object_A> hits <object_B>
2. <object_B> knocks <object_C>
OUTCOME: <final state>
```
**Scoring:** Outcome match (0.5) + intermediate steps (0.5)

In [ ]:
task_type = 'causal_chain'
counts    = SCENARIO_COUNTS[task_type]
task_seed = _task_seed(task_type, MASTER_SEED)
rng       = np.random.RandomState(task_seed)

scenarios_t3 = []
for difficulty in ['easy', 'medium', 'hard']:
    target = counts[difficulty]
    collected, attempt = [], 0
    while len(collected) < target and attempt < target * 3:
        seed     = int(rng.randint(0, 2**31))
        template = CAUSAL_CHAIN_TEMPLATES[attempt % len(CAUSAL_CHAIN_TEMPLATES)]
        attempt += 1
        try:
            s = template.generate(difficulty, seed)
        except Exception:
            continue
        if validate_scenario(s, task_type):
            s['_ground_truth'] = compute_ground_truth(s, task_type)
            collected.append(s)
    scenarios_t3.extend(collected)
    print(f'  T3 {difficulty}: {len(collected)}/{target}')

rows = []
for s in scenarios_t3:
    gt = s['_ground_truth']
    for fmt in ['json', 'ascii', 'nl']:
        rows.append({
            'scenario_id':    s['id'],
            'difficulty':     s['difficulty'],
            'representation': fmt,
            'prompt':         build_prompt(s, fmt, task_type),
            'ground_truth':   json.dumps(gt),
        })
df_causal = pd.DataFrame(rows)
print(f'df_causal: {df_causal.shape}')

In [ ]:
_INTERACTION_KWS = {
    'collision', 'collide', 'hit', 'hits', 'strike', 'impact',
    'launch', 'fall', 'falls', 'drop', 'push', 'slide', 'roll',
    'tip', 'topple', 'bounce', 'knock', 'trigger', 'catapult',
}


def _parse_causal(response: str) -> dict:
    lines = response.strip().split('\n')
    steps, outcome, in_steps = [], '', False
    for line in lines:
        ls = line.strip()
        if re.match(r'^STEPS\s*:', ls, re.IGNORECASE):
            in_steps = True; continue
        if re.match(r'^OUTCOME\s*:', ls, re.IGNORECASE):
            in_steps = False
            outcome = re.sub(r'^OUTCOME\s*:\s*', '', ls, flags=re.IGNORECASE)
            continue
        if in_steps and re.match(r'^\d+\.', ls):
            steps.append(re.sub(r'^\d+\.\s*', '', ls))
    if not steps:
        for line in lines:
            m = re.match(r'^[\d]+[.):]\s*(.+)', line.strip())
            if m:
                steps.append(m.group(1))
    if not outcome:
        outcome = next((l.strip() for l in reversed(lines) if l.strip()), '')
    return {'steps': steps, 'outcome': outcome}


def _outcome_ok(pred: str, actual: str) -> bool:
    pt = set(re.sub(r'[^a-z0-9]', ' ', pred.lower()).split())
    at = set(re.sub(r'[^a-z0-9]', ' ', actual.lower()).split())
    return bool(at) and len(pt & at) / len(at) >= 0.4


def _score_causal(response: str, ground_truth_json: str) -> float:
    gt    = json.loads(ground_truth_json)
    p     = _parse_causal(response)
    score = 0.5 if p['outcome'] and _outcome_ok(p['outcome'], gt.get('outcome', '')) else 0.0
    evts  = gt.get('events', [])
    if evts and p['steps']:
        matched, used = 0, set()
        for step in p['steps']:
            step_lower = step.lower()
            for i, ev in enumerate(evts):
                if i not in used:
                    has_kw  = any(kw in step_lower for kw in _INTERACTION_KWS)
                    has_obj = any(oid.lower() in step_lower for oid in ev.get('objects', []))
                    if has_kw and has_obj:
                        matched += 1; used.add(i); break
        score += 0.5 * (matched / len(evts))
    return min(score, 1.0)


@kbench.task(
    name='PhysIQ Causal Chain Reasoning',
    description=(
        'Given a multi-body 2D physics scenario, describe the chain of causal events '
        'and the final outcome. Tests multi-step temporal reasoning in physics.'
    ),
)
def physiq_causal_chain(llm: kbench.LLMChat, prompt: str, ground_truth: str) -> float:
    """Predict causal event chain in multi-body 2D physics scenario."""
    return _score_causal(llm.prompt(prompt), ground_truth)


print('T3 task defined OK')

In [ ]:
if llm is not None:
    rows = []
    for _, row in df_causal.iterrows():
        run = physiq_causal_chain.run(llm=llm, prompt=row['prompt'], ground_truth=row['ground_truth'])
        rows.append({'scenario_id': row['scenario_id'], 'difficulty': row['difficulty'],
                     'representation': row['representation'], 'score': run.result})
    df_t3 = pd.DataFrame(rows)
    df_t3.to_csv(f'{OUTPUTS_DIR}/task3_causal_chain_results.csv', index=False)
    print(f'T3 mean: {df_t3["score"].mean():.3f}')
else:
    df_t3 = None
    print('Skipped.')

---
## Task 4: Tool Use Planning (Multi-Turn)

Given a 2D physics environment and a goal, interact with the simulation (≤10 turns) using PLACE/PUSH/REMOVE actions to achieve the goal.

**Scoring:** Goal achieved (0.6) + efficiency (0.2) + reasoning quality (0.2)

In [ ]:
task_type = 'tool_use'
counts    = SCENARIO_COUNTS[task_type]
task_seed = _task_seed(task_type, MASTER_SEED)
rng       = np.random.RandomState(task_seed)

scenarios_t4 = []
for difficulty in ['easy', 'medium', 'hard']:
    target = counts[difficulty]
    collected, attempt = [], 0
    while len(collected) < target and attempt < target * 3:
        seed     = int(rng.randint(0, 2**31))
        template = TOOL_USE_TEMPLATES[attempt % len(TOOL_USE_TEMPLATES)]
        attempt += 1
        try:
            s = template.generate(difficulty, seed)
        except Exception:
            continue
        if validate_scenario(s, task_type):
            collected.append(s)
    scenarios_t4.extend(collected)
    print(f'  T4 {difficulty}: {len(collected)}/{target}')

rows = []
for s in scenarios_t4:
    for fmt in ['json', 'ascii', 'nl']:
        s_copy = dict(s); s_copy['_format'] = fmt
        rows.append({
            'scenario_id':    s['id'],
            'difficulty':     s['difficulty'],
            'representation': fmt,
            'scenario_json':  json.dumps(s_copy),
        })
df_tool_use = pd.DataFrame(rows)
print(f'df_tool_use: {df_tool_use.shape}')

In [ ]:
# ── Shared multi-turn helpers (Tasks 4 & 5) ───────────────────────────────────────

def _fmt(scenario: dict, fmt: str) -> str:
    if fmt == 'ascii': return format_as_ascii(scenario)
    if fmt == 'nl':    return format_as_nl(scenario)
    return format_as_json(scenario)


_PHYSICS_KWS = {
    'force', 'gravity', 'friction', 'momentum', 'velocity', 'mass',
    'inertia', 'torque', 'lever', 'ramp', 'slope', 'balance', 'weight',
    'support', 'bridge', 'push', 'angle', 'trajectory', 'arc',
}


def _reasoning_valid(response: str) -> bool:
    return sum(1 for kw in _PHYSICS_KWS if kw in response.lower()) >= 3


def _parse_action(response: str):
    """Extract ACTION: line → dict, or None if unparseable."""
    m = re.search(r'^ACTION\s*:\s*(.+)$', response, re.IGNORECASE | re.MULTILINE)
    if not m:
        return None
    a = m.group(1).strip()
    pm = re.match(
        r'PLACE\s+(\w+)\s+AT\s+([\d.\-]+)\s+([\d.\-]+)(?:\s+ANGLE\s+([\d.\-]+))?',
        a, re.IGNORECASE
    )
    if pm:
        return {'type': 'place', 'object_type': pm.group(1),
                'position': [float(pm.group(2)), float(pm.group(3))],
                'angle': float(pm.group(4)) if pm.group(4) else 0.0}
    pu = re.match(
        r'PUSH\s+(\w+)\s+(?:WITH_FORCE|FORCE)\s+([\d.\-]+)\s+DIRECTION\s+([\d.\-]+)',
        a, re.IGNORECASE
    )
    if pu:
        return {'type': 'push', 'object_id': pu.group(1),
                'force': float(pu.group(2)), 'direction': float(pu.group(3))}
    rm = re.match(r'REMOVE\s+(\w+)', a, re.IGNORECASE)
    if rm:
        return {'type': 'remove', 'object_id': rm.group(1)}
    return None


print('Multi-turn helpers defined OK')

In [ ]:
@kbench.task(
    name='PhysIQ Tool Use Planning',
    description=(
        'Use \u226410 actions (PLACE/PUSH/REMOVE) to manipulate a 2D physics world and '
        'achieve the stated goal. Tests means-end reasoning and creative problem solving.'
    ),
)
def physiq_tool_use(llm: kbench.LLMChat, scenario_json: str) -> float:
    """Multi-turn: propose physics actions to achieve a goal."""
    scenario  = json.loads(scenario_json)
    fmt       = scenario.pop('_format', 'json')
    world     = PhysIQWorld(scenario)
    goal      = scenario.get('goal')
    goal_desc = scenario.get('goal_description', str(goal))
    tools     = scenario.get('available_tools', [])
    tool_list = '\n'.join(f'  - {t}' for t in tools) or '  (none specified)'
    turns_used, any_valid_reasoning = 0, False

    prompt = (
        f'You are controlling a 2D physics simulation.\n\nGOAL: {goal_desc}\n\n'
        f'AVAILABLE TOOLS:\n{tool_list}\n\nCURRENT STATE:\n{_fmt(scenario, fmt)}\n\n'
        f'Actions:\n  PLACE <type> AT <x> <y> ANGLE <deg>\n'
        f'  PUSH <id> FORCE <n> DIRECTION <deg>\n  REMOVE <id>\n\n'
        f'Explain your reasoning, then output: ACTION: ...'
    )
    for _turn in range(MAX_TURNS):
        response   = llm.prompt(prompt)
        turns_used += 1
        if _reasoning_valid(response):
            any_valid_reasoning = True
        action = _parse_action(response)
        if action is None:
            prompt = (
                f"Couldn't parse action. Use one of:\n"
                f'  ACTION: PLACE <type> AT <x> <y> ANGLE <deg>\n'
                f'  ACTION: PUSH <id> FORCE <n> DIRECTION <deg>\n'
                f'  ACTION: REMOVE <id>\n'
                f'Turns remaining: {MAX_TURNS - turns_used}'
            )
            continue
        world.execute_action(action)
        world.simulate(0.5)
        if world.check_goal(goal):
            return score_tool_use(goal_achieved=True, turns_used=turns_used,
                                   max_turns=MAX_TURNS, reasoning_valid=any_valid_reasoning)
        if turns_used >= MAX_TURNS:
            break
        prompt = (
            f'UPDATED STATE:\n{_fmt(scenario, fmt)}\n\nGOAL: {goal_desc}\n'
            f'Turns remaining: {MAX_TURNS - turns_used}\n\nACTION: ...'
        )
    return score_tool_use(goal_achieved=False, turns_used=MAX_TURNS,
                           max_turns=MAX_TURNS, reasoning_valid=any_valid_reasoning,
                           progress=world.measure_progress(goal))


print('physiq_tool_use defined OK')

In [ ]:
if llm is not None:
    rows = []
    for _, row in df_tool_use.iterrows():
        run = physiq_tool_use.run(llm=llm, scenario_json=row['scenario_json'])
        rows.append({'scenario_id': row['scenario_id'], 'difficulty': row['difficulty'],
                     'representation': row['representation'], 'score': run.result})
    df_t4 = pd.DataFrame(rows)
    df_t4.to_csv(f'{OUTPUTS_DIR}/task4_tool_use_results.csv', index=False)
    print(f'T4 mean: {df_t4["score"].mean():.3f}')
else:
    df_t4 = None
    print('Skipped.')

---
## Task 5: Adaptive Replanning (Multi-Turn)

Same as Tool Use Planning, but a **perturbation is injected** after the first successfully parsed action — a material changes, a support breaks, an object drifts, a tool disappears, or a new obstacle appears. The model must recognise the failure and adapt.

**Scoring:** Failure recognition (0.2) + recovery plan (0.3) + goal achieved (0.3) + efficiency (0.2)

In [ ]:
task_type = 'replan'
counts    = SCENARIO_COUNTS[task_type]
task_seed = _task_seed(task_type, MASTER_SEED)
rng       = np.random.RandomState(task_seed)

scenarios_t5 = []
for difficulty in ['easy', 'medium', 'hard']:
    target = counts[difficulty]
    collected, attempt = [], 0
    while len(collected) < target and attempt < target * 3:
        seed     = int(rng.randint(0, 2**31))
        template = REPLAN_TEMPLATES[attempt % len(REPLAN_TEMPLATES)]
        attempt += 1
        try:
            s = template.generate(difficulty, seed)
        except Exception:
            continue
        if validate_scenario(s, task_type):
            collected.append(s)
    scenarios_t5.extend(collected)
    print(f'  T5 {difficulty}: {len(collected)}/{target}')

rows = []
for s in scenarios_t5:
    for fmt in ['json', 'ascii', 'nl']:
        s_copy = dict(s); s_copy['_format'] = fmt
        rows.append({
            'scenario_id':    s['id'],
            'difficulty':     s['difficulty'],
            'representation': fmt,
            'scenario_json':  json.dumps(s_copy),
        })
df_replan = pd.DataFrame(rows)
print(f'df_replan: {df_replan.shape}')

In [ ]:
_RECOGNITION_KWS = {
    'unexpected', 'changed', 'different', 'surprise', 'noticed',
    'heavier', 'broken', 'missing', 'new obstacle', 'shifted', 'moved',
    'drifted', 'disappeared', 'failed', 'did not work',
}
_RECOVERY_KWS = {
    'instead', 'alternative', 'adjust', 'change', 'modify', 'different',
    'adapt', 'replan', 'revise', 'rethink', 'new approach', 'try again',
    'new plan', 'different strategy',
}


def _recognizes_failure(r: str) -> bool:
    return any(kw in r.lower() for kw in _RECOGNITION_KWS)


def _has_recovery_plan(r: str) -> bool:
    return sum(1 for kw in _RECOVERY_KWS if kw in r.lower()) >= 2


@kbench.task(
    name='PhysIQ Adaptive Replanning',
    description=(
        'Use \u226410 actions to achieve a goal in a 2D physics world. '
        'A surprise perturbation is injected after the first action. '
        'Tests cognitive flexibility and error recovery.'
    ),
)
def physiq_replan(llm: kbench.LLMChat, scenario_json: str) -> float:
    """Multi-turn with forced perturbation after the first parsed action."""
    scenario     = json.loads(scenario_json)
    fmt          = scenario.pop('_format', 'json')
    world        = PhysIQWorld(scenario)
    goal         = scenario.get('goal')
    goal_desc    = scenario.get('goal_description', str(goal))
    perturbation = scenario.get('perturbation')
    tools        = scenario.get('available_tools', [])
    tool_list    = '\n'.join(f'  - {t}' for t in tools) or '  (none specified)'

    turns_used = 0
    perturbed  = False
    failure_recognized = recovery_plan_valid = False
    recovery_turns = 0

    prompt = (
        f'GOAL: {goal_desc}\n\nAVAILABLE TOOLS:\n{tool_list}\n\n'
        f'STATE:\n{_fmt(scenario, fmt)}\n\n'
        f'Actions: PLACE <type> AT <x> <y> ANGLE <deg> | PUSH <id> FORCE <n> DIRECTION <deg> | REMOVE <id>\n\n'
        f'Explain your reasoning, then: ACTION: ...'
    )
    for _turn in range(MAX_TURNS):
        response    = llm.prompt(prompt)
        turns_used += 1
        if perturbed:
            if _recognizes_failure(response): failure_recognized  = True
            if _has_recovery_plan(response):  recovery_plan_valid = True
            recovery_turns += 1
        action = _parse_action(response)
        if action is None:
            prompt = f'Parse error. Try again. Turns remaining: {MAX_TURNS - turns_used}\nACTION: ...'
            continue
        world.execute_action(action)
        world.simulate(0.5)
        if not perturbed and perturbation:
            world.apply_perturbation(perturbation)
            perturbed = True
            desc   = perturbation.get('description', 'Something unexpected happened.')
            prompt = (
                f'PERTURBATION: {desc}\n\nSTATE:\n{_fmt(scenario, fmt)}\n\n'
                f'GOAL: {goal_desc}\n'
                f'Your previous plan may not work. Describe what changed, then propose a revised action.\n'
                f'Turns remaining: {MAX_TURNS - turns_used}\nACTION: ...'
            )
            continue
        if world.check_goal(goal):
            return score_replan(failure_recognized=failure_recognized,
                                recovery_plan_valid=recovery_plan_valid,
                                goal_achieved=True, recovery_turns=recovery_turns)
        if turns_used >= MAX_TURNS:
            break
        prompt = (
            f'STATE:\n{_fmt(scenario, fmt)}\n\nGOAL: {goal_desc}\n'
            f'Turns remaining: {MAX_TURNS - turns_used}\nACTION: ...'
        )
    return score_replan(failure_recognized=failure_recognized,
                        recovery_plan_valid=recovery_plan_valid,
                        goal_achieved=False, recovery_turns=recovery_turns)


print('physiq_replan defined OK')

In [ ]:
if llm is not None:
    rows = []
    for _, row in df_replan.iterrows():
        run = physiq_replan.run(llm=llm, scenario_json=row['scenario_json'])
        rows.append({'scenario_id': row['scenario_id'], 'difficulty': row['difficulty'],
                     'representation': row['representation'], 'score': run.result})
    df_t5 = pd.DataFrame(rows)
    df_t5.to_csv(f'{OUTPUTS_DIR}/task5_replanning_results.csv', index=False)
    print(f'T5 mean: {df_t5["score"].mean():.3f}')
else:
    df_t5 = None
    print('Skipped.')

---
## Aggregate Results

Collects all task results and computes the overall **PhysIQ Score**, **Format Robustness Score (FRS)**, and difficulty scaling.

In [ ]:
task_dfs = {
    'trajectory':   df_t1,
    'stability':    df_t2,
    'causal_chain': df_t3,
    'tool_use':     df_t4,
    'replanning':   df_t5,
}
available = {k: v for k, v in task_dfs.items() if v is not None}
print(f'{len(available)}/5 tasks have results')

if available:
    task_means = {k: v['score'].mean() for k, v in available.items()}
    physiq     = sum(task_means[k] * TASK_WEIGHTS.get(k, 0.2) for k in task_means)
    print(f'\nPhysIQ Score: {physiq:.3f}')
    for k, v in task_means.items():
        print(f'  {k}: {v:.3f}')

    # Format Robustness Score
    all_frs = []
    for df in available.values():
        for sid, grp in df.groupby('scenario_id'):
            scores = grp['score'].values
            mx     = scores.max()
            all_frs.append(1.0 - (mx - scores.min()) / mx if mx > 0 else 1.0)
    frs = float(np.mean(all_frs))
    print(f'\nFormat Robustness Score (FRS): {frs:.3f}')
    print('  1.0 = perfectly consistent across JSON / ASCII / NL')

    # Difficulty scaling
    all_df = pd.concat(list(available.values()), ignore_index=True)
    if 'difficulty' in all_df.columns:
        print('\nDifficulty scaling (should decrease):')
        dm = all_df.groupby('difficulty')['score'].mean()
        for d in ['easy', 'medium', 'hard']:
            if d in dm:
                print(f'  {d}: {dm[d]:.3f}')

In [ ]:
if available:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Per-task scores
    pd.Series(task_means).plot(kind='bar', ax=axes[0], color='#2196F3')
    axes[0].set_title(f'PhysIQ Score per Task  (overall={physiq:.3f})')
    axes[0].set_ylabel('Score (0–1)')
    axes[0].set_ylim(0, 1)
    axes[0].axhline(physiq, color='red', linestyle='--', alpha=0.7, label='PhysIQ')
    axes[0].legend()
    plt.setp(axes[0].get_xticklabels(), rotation=30, ha='right')

    # Format breakdown
    all_df    = pd.concat(list(available.values()), ignore_index=True)
    fmt_means = all_df.groupby('representation')['score'].mean()
    fmt_means.plot(kind='bar', ax=axes[1], color=['#2196F3', '#9C27B0', '#00BCD4'])
    axes[1].set_title(f'Score by Format  (FRS={frs:.3f})')
    axes[1].set_ylabel('Mean Score (0–1)')
    axes[1].set_ylim(0, 1)
    plt.setp(axes[1].get_xticklabels(), rotation=0)

    # Difficulty
    if 'difficulty' in all_df.columns:
        all_df.groupby('difficulty')['score'].mean().reindex(['easy', 'medium', 'hard']).plot(
            kind='bar', ax=axes[2], color=['#4CAF50', '#FF9800', '#F44336']
        )
        axes[2].set_title('Score by Difficulty')
        axes[2].set_ylabel('Mean Score (0–1)')
        axes[2].set_ylim(0, 1)
        plt.setp(axes[2].get_xticklabels(), rotation=0)

    plt.suptitle('PhysIQ Benchmark — Aggregate Results', size=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUTS_DIR}/physiq_aggregate.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {OUTPUTS_DIR}/physiq_aggregate.png')

In [ ]:
# Register all 5 tasks with the Kaggle benchmarks runner
%choose physiq_trajectory
%choose physiq_stability
%choose physiq_causal_chain
%choose physiq_tool_use
%choose physiq_replan